# RAGLedger: real-data results on Colab GPU

Everything this notebook measures uses **published attacks and public data**:

| piece | source | status |
|---|---|---|
| poisoned passages | the PoisonedRAG authors' published `adv_targeted_results/nq.json`, injected **verbatim** | real, unmodified |
| corpus / queries / gold passages | BEIR Natural Questions | real |
| retriever, NLI, generator | Contriever, DeBERTa-v3 NLI, Qwen2.5-7B (Ollama) or Llama-3.1-8B (vLLM) | real models, pinned |
| TrustRAG baseline (`trustrag_official`) | the authors' own `k_mean_filtering` code, stage 1 only | real code, partial method |
| other baselines | label-free **adaptations** of TrustRAG/RobustRAG ideas, simple filters, ablations | adapted, labelled as such |

Nothing here is simulated: no stub LLM, no hashing embedder, no generated poison for the headline attacks.
Template attacks (`adaptive`, `slow_burn`, ...) are this project's own and are labelled *own attacks*.

**Runtime:** GPU. A free T4 can run the Qwen/Ollama path. The Llama/vLLM path needs an L4/A100 (bfloat16).
Run the cells top to bottom; every long step resumes after a disconnect.

## 1. GPU check

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time, urllib.request, hashlib

def run(cmd, **kw):
    return subprocess.run(cmd, text=True, capture_output=True, check=False, **kw)

gpu = run(["nvidia-smi", "-L"])
print(gpu.stdout or gpu.stderr)
if not (gpu.returncode == 0 and "GPU" in gpu.stdout):
    raise RuntimeError("No CUDA GPU. Runtime -> Change runtime type -> GPU, then rerun.")
print("disk free (GB):", round(shutil.disk_usage("/content").free / 1e9, 1))

## 2. Clone this repository at an exact commit

In [ ]:
REPO_URL = "https://github.com/athishss/-Adaptive-Trust-Based-Framework-for-Detecting-and-Preventing-Knowledge-Base-Poisoning-in-RAG-Systems.git"
PINNED_COMMIT = "71e97cf697b2628a0588caa2d809cdbdfb6f3ecc"
REPO = pathlib.Path("/content/RAGLedger")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--quiet"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", PINNED_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == PINNED_COMMIT, (actual, PINNED_COMMIT)
print("checked-out SHA:", actual)

## 3. Google Drive (results) and local scratch (indexes, caches)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/RAGLedger_real")
ACTIVE = pathlib.Path("/content/ragledger_active")
for p in (DRIVE / "datasets", DRIVE / "exports", DRIVE / "logs", DRIVE / "configs", ACTIVE / "runs", ACTIVE / "preflight"):
    p.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
print("results persist in:", DRIVE)

## 4. Install

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all,data]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk", "rouge_score"], check=True)  # TrustRAG's filter
import torch, transformers, faiss, trace_rag
assert torch.cuda.is_available(), "PyTorch cannot see the GPU"
VERSIONS = {"torch": torch.__version__, "transformers": transformers.__version__,
            "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda}
print(json.dumps(VERSIONS, indent=2))
assert str(trace_rag.__file__).startswith(str(REPO))

## 5. External reference code (cloned read-only, pinned)

* PoisonedRAG (MIT): supplies the published poison. Pin `f660d72174f0`.
* TrustRAG (MIT): supplies `defend_module.py`, called unmodified by `trustrag_official`. Pin `11dcea0262d1`.

Nothing from these repositories is copied into this project.

In [ ]:
EXT = pathlib.Path("/content/external"); EXT.mkdir(exist_ok=True)
PINS = {"PoisonedRAG": ("https://github.com/sleeepeer/PoisonedRAG.git", "f660d72174f06b13fae5163ce656e7b235db858f"),
        "TrustRAG": ("https://github.com/HuichiZhou/TrustRAG.git", "11dcea0262d14b0e38e22e1d7ddce4a151e82a61")}
for name, (url, sha) in PINS.items():
    d = EXT / name
    if not (d / ".git").exists():
        subprocess.run(["git", "clone", "--quiet", url, str(d)], check=True)
    subprocess.run(["git", "-C", str(d), "checkout", "--quiet", sha], check=True)
    got = subprocess.check_output(["git", "-C", str(d), "rev-parse", "HEAD"], text=True).strip()
    assert got == sha, (name, got)
    print(name, got)
POISON_JSON = EXT / "PoisonedRAG/results/adv_targeted_results/nq.json"
TRUSTRAG_DIR = EXT / "TrustRAG"
assert POISON_JSON.exists() and (TRUSTRAG_DIR / "defend_module.py").exists()
records = json.loads(POISON_JSON.read_text())
print("published poison records:", len(records))

## 6. Cache the pinned embedder and NLI model

In [ ]:
from huggingface_hub import snapshot_download
EMBED_ID, EMBED_REV = "facebook/contriever", "2bd46a25019aeea091fd42d1f0fd4801675cf699"
NLI_ID, NLI_REV = "cross-encoder/nli-deberta-v3-base", "6c749ce3425cd33b46d187e45b92bbf96ee12ec7"
for model, rev in ((EMBED_ID, EMBED_REV), (NLI_ID, NLI_REV)):
    print(model, rev, "->", snapshot_download(model, revision=rev))

## 7. Generator

Choose one. **Do not pool numbers across generators**; each choice is its own experiment.

* `qwen_ollama`: Qwen2.5-7B q4_K_M via Ollama; fits a free T4.
* `llama_vllm`: Llama-3.1-8B-Instruct bfloat16 via vLLM; needs an L4/A100 and a Hugging Face token with Llama access
  (Colab secret `HF_TOKEN`).

In [ ]:
GENERATOR = "qwen_ollama"      # or "llama_vllm"
LLM_CACHE = ACTIVE / "llm_cache.sqlite"   # greedy decoding only; hits still count as calls and keep original latency

if GENERATOR == "qwen_ollama":
    OLLAMA_VERSION = "0.5.7"
    if not shutil.which("zstd"):
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
    if not shutil.which("ollama"):
        archive = pathlib.Path("/tmp/ollama-linux-amd64.tgz")
        subprocess.run(["curl", "-fL", f"https://github.com/ollama/ollama/releases/download/v{OLLAMA_VERSION}/ollama-linux-amd64.tgz", "-o", str(archive)], check=True)
        subprocess.run(["tar", "-xzf", str(archive), "-C", "/usr/local"], check=True)
    def ollama_up():
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read(); return True
        except Exception:
            return False
    if not ollama_up():
        subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "ab"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_up(): break
        time.sleep(2)
    assert ollama_up(), "Ollama did not start; see /tmp/ollama.log"
    MODEL = "qwen2.5:7b"
    subprocess.run(["ollama", "pull", MODEL], check=True)
    tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))
    DIGEST = next(m for m in tags["models"] if m["name"] == MODEL)["digest"]
    req = urllib.request.Request("http://127.0.0.1:11434/api/show", data=json.dumps({"name": MODEL}).encode(),
                                 headers={"Content-Type": "application/json"})
    QUANT = json.load(urllib.request.urlopen(req)).get("details", {}).get("quantization_level", "UNSPECIFIED")
    assert DIGEST.startswith("sha256:") and QUANT != "UNSPECIFIED", (DIGEST, QUANT)
    CONFIG = REPO / "config/colab_pilot_qwen.yaml"
    GEN_OVERRIDES = [f"generation.revision={DIGEST}", f"generation.quantization={QUANT}",
                     f"generation.serving_version={OLLAMA_VERSION}"]
    GEN_LOCK = {"generator": "qwen2.5:7b via Ollama", "digest": DIGEST, "quantization": QUANT, "serving": OLLAMA_VERSION}
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    LLAMA_ID, LLAMA_REV, VLLM_VERSION = "meta-llama/Llama-3.1-8B-Instruct", "a09a35458c702b33eeacc393d103063234e8bc28", "0.7.3"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"vllm=={VLLM_VERSION}"], check=True)
    subprocess.Popen(["vllm", "serve", LLAMA_ID, "--revision", LLAMA_REV, "--port", "8000", "--dtype", "bfloat16",
                      "--gpu-memory-utilization", "0.55"], stdout=open(DRIVE / "logs" / "vllm.log", "ab"), stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2).read(); break
        except Exception:
            time.sleep(5)
    else:
        raise RuntimeError("vLLM did not come up; see Drive/logs/vllm.log")
    CONFIG = REPO / "config/nq_gpu.yaml"
    GEN_OVERRIDES = [f"generation.revision={LLAMA_REV}", "generation.backend=openai", f"generation.model_name={LLAMA_ID}",
                     "generation.precision=bfloat16", f"generation.serving_version={VLLM_VERSION}",
                     "generation.base_url=http://127.0.0.1:8000/v1"]
    GEN_LOCK = {"generator": LLAMA_ID, "revision": LLAMA_REV, "serving": f"vLLM {VLLM_VERSION}"}

OVERRIDES = [f"embedding.revision={EMBED_REV}", f"security.nli_revision={NLI_REV}",
             f"generation.cache_path={LLM_CACHE}"] + GEN_OVERRIDES
print(json.dumps(GEN_LOCK, indent=2))

## 8. Real data: BEIR NQ subset that contains the published questions

`download_data.py --include-queries` forces the 100 PoisonedRAG questions (and their gold passages) into the subset.
The raw NQ corpus is several GB, so downloading is an **explicit opt-in**: set `RUN_DATASET_DOWNLOAD = True` after checking Drive space.

In [ ]:
RUN_DATASET_DOWNLOAD = False
SUBSET_PASSAGES = 100_000
DATA = DRIVE / "datasets" / "nq_real"
CORPUS, QUERIES, QRELS = DATA / "corpus_subset.parquet", DATA / "queries_subset.jsonl", DATA / "qrels.tsv"
if not all(p.exists() for p in (CORPUS, QUERIES, QRELS)):
    if not RUN_DATASET_DOWNLOAD:
        raise RuntimeError("Dataset missing. Set RUN_DATASET_DOWNLOAD=True (several GB) or copy the files into " + str(DATA))
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, str(REPO / "scripts/download_data.py"), "--dataset", "nq", "--out", str(DATA),
                    "--subset", str(SUBSET_PASSAGES), "--queries", "500", "--include-queries", str(POISON_JSON)],
                   cwd=REPO, check=True)
assert all(p.exists() for p in (CORPUS, QUERIES, QRELS))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
DATA_HASHES = {"poison_json": sha(POISON_JSON), "corpus_subset": sha(CORPUS), "qrels": sha(QRELS)}
print(json.dumps(DATA_HASHES, indent=2))

## 9. Experiment design and dataset check

Every seed attacks a **disjoint block** of published questions, so seeds are independent samples of questions
(this is what the confidence intervals and paired tests assume). The scorer is trained on a further disjoint block.
Questions whose gold passage is missing are dropped and counted, never replaced.

In [ ]:
TIER = "smoke"   # "smoke" (minutes) -> "main" (hours) -> "full"
TIERS = {
  "smoke": dict(attacks=["poisonedrag_official"], baselines=["no_defence", "TrustRAG", "trustrag_official", "RAGLedger"],
                seeds=[20260921], targets=10),
  "main":  dict(attacks=["poisonedrag_official", "poisonedrag_official_sybil"],
                baselines=["no_defence", "duplicate_filter", "trust_threshold", "perplexity", "suspicion_only",
                           "corroboration_only", "trust_no_deferred", "TrustRAG", "RobustRAG", "trustrag_official",
                           "always_on_loo", "RAGLedger"],
                seeds=[20260921, 20260922, 20260923], targets=20),
  "full":  dict(attacks=["poisonedrag_official", "poisonedrag_official_sybil", "coordinated", "adaptive", "slow_burn", "hit_and_run"],
                baselines=["no_defence", "duplicate_filter", "trust_threshold", "perplexity", "suspicion_only",
                           "corroboration_only", "trust_no_deferred", "TrustRAG", "RobustRAG", "trustrag_official",
                           "always_on_loo", "RAGLedger"],
                seeds=[20260921, 20260922, 20260923], targets=20),
}
T = TIERS[TIER]
N_EVAL = len(T["seeds"]) * T["targets"]
TRAIN_TARGETS, MAX_CLEAN = 20, 3000
from trace_rag.evaluation.poisonedrag import load_poisonedrag_fixture
_, fx_report = load_poisonedrag_fixture(POISON_JSON, CORPUS, QRELS, target_limit=1, max_clean=50)
print(json.dumps(fx_report, indent=2))
need = N_EVAL + TRAIN_TARGETS + 1
assert fx_report["kept"] >= need, f"{fx_report['kept']} usable questions < {need} needed; lower seeds/targets or use TIER='smoke'"
print(f"design: {len(T['attacks'])} attacks x {len(T['baselines'])} defences x {len(T['seeds'])} seeds, "
      f"{T['targets']} targets per seed, scorer trained on questions [{N_EVAL}, {N_EVAL + TRAIN_TARGETS})")

## 10. Functional preflight (real models, isolated storage)

In [ ]:
PREFLIGHT_PARENT = ACTIVE / "preflight"
def preflight(config, overrides, label):
    cmd = [sys.executable, str(REPO / "scripts/check_research_environment.py"), "--config", str(config), "--functional", "--json",
           "--preflight-storage-root", str(PREFLIGHT_PARENT / label), "--corpus", str(CORPUS), "--queries", str(QUERIES), "--qrels", str(QRELS)]
    for item in overrides + [f"storage.root={ACTIVE / 'runs' / label}"]:
        cmd += ["--set", item]
    proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
    (DRIVE / "logs" / f"preflight_{label}.log").write_text(proc.stdout + proc.stderr)
    payload = json.loads(proc.stdout[proc.stdout.find("{"):])
    print(label, payload["status"], payload["status_reason"])
    if payload["status"] != "READY":
        print(json.dumps(payload.get("failures", []), indent=2))
        raise RuntimeError(f"preflight is {payload['status']}, not READY")
    return payload
PREFLIGHT = preflight(CONFIG, OVERRIDES, "real")

## 11. Sanity gate: does the undefended pipeline reproduce the published attack?

With no defence the published poison should be retrieved and should change the answer. Compare the ASR printed here with the ASR the
PoisonedRAG paper reports for a comparable generator. The thresholds below only catch broken plumbing (wrong corpus, wrong retriever);
they are **not** a published criterion.

In [ ]:
GATE = ACTIVE / "runs" / "gate"
cmd = [sys.executable, str(REPO / "scripts/run_poisonedrag_official.py"), "--poison", str(POISON_JSON), "--corpus", str(CORPUS),
       "--qrels", str(QRELS), "--config", str(CONFIG), "--out", str(GATE), "--targets", "10", "--max-clean", str(MAX_CLEAN),
       "--attacks", "poisonedrag_official", "--baselines", "no_defence", "--seeds", str(T["seeds"][0])]
for item in OVERRIDES: cmd += ["--set", item]
t0 = time.time(); proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True); print("elapsed s:", round(time.time() - t0))
(DRIVE / "logs" / "gate.log").write_text(proc.stdout + proc.stderr)
if proc.returncode != 0: print(proc.stdout[-3000:], proc.stderr[-3000:]); raise RuntimeError("gate run failed")
gate = json.loads((GATE / "results.json").read_text())["results"][0]
print({k: gate[k] for k in ("poison_exposure_rate", "answer_level_asr", "answer_level_asr_status", "eligible_attack_attempts")})
assert gate["answer_level_asr_status"] == "MEASURED"
assert gate["poison_exposure_rate"] >= 0.5, "published poison is not reaching the context without a defence: fix retrieval before continuing"
assert gate["answer_level_asr"] >= 0.3, "undefended ASR is far below the published attack: check generator, prompt and answer matching"

## 12. Train the suspicion scorer on *different* real questions and *different* attacks

Training poison comes from this project's template attacks on a disjoint block of real NQ questions. The published PoisonedRAG poison is
never used for training: the evaluation therefore tests **generalisation to an attack the scorer has not seen**.

In [ ]:
SCORER = ACTIVE / "runs" / "scorer.joblib"
cmd = [sys.executable, str(REPO / "scripts/train_suspicion_scorer.py"), "--config", str(CONFIG), "--out", str(SCORER),
       "--poisonedrag-json", str(POISON_JSON), "--corpus", str(CORPUS), "--qrels", str(QRELS),
       "--targets", str(TRAIN_TARGETS), "--target-offset", str(N_EVAL), "--max-clean", str(MAX_CLEAN),
       "--attacks", "black_box,white_box,entity_swap,sybil,coordinated,adaptive"]
for item in OVERRIDES: cmd += ["--set", item]
proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
(DRIVE / "logs" / "scorer_training.log").write_text(proc.stdout + proc.stderr)
if proc.returncode != 0: print(proc.stdout[-3000:], proc.stderr[-3000:]); raise RuntimeError("scorer training failed")
print(proc.stdout[-2500:])
shutil.copy2(SCORER, DRIVE / "configs" / "scorer.joblib")
OVERRIDES_MATRIX = OVERRIDES + [f"scorer.model_path={SCORER}"]

## 13. The experiment matrix

Resumable: each finished case writes `result.json`; re-running this cell skips finished cases.
`TIER="smoke"` first to measure seconds per case on your GPU, then switch to `"main"`.

In [ ]:
OUT = ACTIVE / "runs" / f"matrix_{TIER}_{GENERATOR}"
cmd = [sys.executable, str(REPO / "scripts/run_poisonedrag_official.py"), "--poison", str(POISON_JSON), "--corpus", str(CORPUS),
       "--qrels", str(QRELS), "--config", str(CONFIG), "--out", str(OUT), "--targets", str(T["targets"]), "--max-clean", str(MAX_CLEAN),
       "--attacks", ",".join(T["attacks"]), "--baselines", ",".join(T["baselines"]), "--seeds", ",".join(map(str, T["seeds"])),
       "--trustrag-dir", str(TRUSTRAG_DIR), "--scope", f"REAL DATA ({TIER}; {GEN_LOCK['generator']})"]
for item in OVERRIDES_MATRIX: cmd += ["--set", item]
print(" ".join(cmd))
t0 = time.time()
with open(DRIVE / "logs" / f"matrix_{TIER}_{GENERATOR}.log", "ab") as log:
    proc = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
print("exit code:", proc.returncode, "| elapsed min:", round((time.time() - t0) / 60, 1))
summary = json.loads((OUT / "run_summary.json").read_text())
print(json.dumps({k: summary[k] for k in ("cases_completed", "cases_requested", "failures")}, indent=2)[:3000])

## 14. Statistics

Unit = target question. 95% bootstrap CIs; paired exact McNemar between RAGLedger and every other defence, Holm-corrected.
Tables always show **ASR, accuracy under attack and abstention together**: abstention counts as attack failure, so a system that refuses everything has ASR 0.

In [ ]:
proc = subprocess.run([sys.executable, str(REPO / "scripts/analyze_results.py"), str(OUT), "--reference", "RAGLedger",
                       "--scope", f"{TIER}; {GEN_LOCK['generator']}"], cwd=REPO, text=True, capture_output=True)
print(proc.stdout, proc.stderr[-2000:])
from IPython.display import Markdown, display
display(Markdown((OUT / "analysis.md").read_text()))

## 15. Figures

In [ ]:
import matplotlib.pyplot as plt, numpy as np
A = json.loads((OUT / "analysis.json").read_text())
rows = A["summary"]
attacks = sorted({r["attack"] for r in rows}); defences = [b for b in T["baselines"] if any(r["baseline"] == b for r in rows)]
FIG = DRIVE / "exports" / f"figures_{TIER}_{GENERATOR}"; FIG.mkdir(parents=True, exist_ok=True)
for metric, title in (("asr_last", "Attack success rate (final attempt)"), ("accuracy", "Accuracy under attack"), ("abstention", "Abstention rate")):
    fig, axes = plt.subplots(1, len(attacks), figsize=(5.2 * len(attacks), 4), squeeze=False)
    for ax, attack in zip(axes[0], attacks):
        vals, lo, hi = [], [], []
        for d in defences:
            r = next((x for x in rows if x["attack"] == attack and x["baseline"] == d), None)
            v = r[metric] if r and r[metric] is not None else np.nan
            ci = r[f"{metric}_ci"] if r else [None, None]
            vals.append(v); lo.append(v - ci[0] if ci[0] is not None else 0); hi.append(ci[1] - v if ci[1] is not None else 0)
        ax.bar(range(len(defences)), vals, yerr=[lo, hi], capsize=3)
        ax.set_xticks(range(len(defences))); ax.set_xticklabels(defences, rotation=60, ha="right", fontsize=8)
        ax.set_ylim(0, 1); ax.set_title(f"{title}\n{attack}", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG / f"{metric}.png", dpi=200); fig.savefig(FIG / f"{metric}.pdf"); plt.show()
print("figures saved to", FIG)

## 16. Export everything needed to reproduce and write the paper

The manifest records commit SHAs, model digests, dataset hashes and the exact design. The paper must be written **from `analysis.json` only**.

In [ ]:
EXPORT = DRIVE / "exports" / f"{TIER}_{GENERATOR}_{time.strftime('%Y%m%d_%H%M%S')}"; EXPORT.mkdir(parents=True)
for path in OUT.rglob("*"):
    if path.is_file() and "/run/" not in str(path) and path.suffix in (".json", ".jsonl", ".csv", ".md"):
        dest = EXPORT / path.relative_to(OUT); dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(path, dest)
shutil.copy2(LLM_CACHE, EXPORT / "llm_cache.sqlite") if LLM_CACHE.exists() else None
MANIFEST = {"tier": TIER, "design": T, "repo_commit": PINNED_COMMIT, "external": {k: v[1] for k, v in PINS.items()},
            "embedder": {"id": EMBED_ID, "revision": EMBED_REV}, "nli": {"id": NLI_ID, "revision": NLI_REV},
            "generator": GEN_LOCK, "config": str(CONFIG.relative_to(REPO)), "overrides": OVERRIDES_MATRIX,
            "dataset_hashes": DATA_HASHES, "fixture_report": fx_report, "environment": VERSIONS,
            "claims_allowed": "headline attacks use the published PoisonedRAG poison verbatim; adapted baselines are not the published methods"}
(EXPORT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2))
print("exported to", EXPORT)

## What this notebook does *not* do

* **RobustRAG** is a different pipeline (isolate-then-aggregate decoding), so it is reproduced in the companion notebook
  `ragledger_baselines_official.ipynb` on its own benchmark. In this notebook `RobustRAG` is a labelled *adaptation*.
* TrustRAG stage 2 (LLM self-assessment) is not included in `trustrag_official`.
* Defence rows other than `trustrag_official` are **not published methods**; say "adapted" in the paper.